# 🧠 Notebook 08: Model Training & Baseline Benchmarking

> Phase 3 of the FIFA World Cup Predictor project: Train, evaluate, and compare baseline ML models using strict temporal validation and enriched match features.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add src to Python path
sys.path.insert(0, '../src')

from train_model import load_training_data, build_model_pipeline, train_and_evaluate, LABEL_MAP
from evaluate_model import evaluate_saved_model


## 1. Load Processed Training Dataset


In [ ]:
df = load_training_data()
print(f"Total matches loaded: {len(df):,}")
print(f"Date span: {df['date'].min().date()} to {df['date'].max().date()}")
df.head(5)


## 2. Temporal Train / Test Split Breakdown

To prevent future data leakage, we split chronologically: training on matches through 2020, and evaluating on modern matches from 2021 through 2026.


In [ ]:
train_mask = df['date'].dt.year <= 2020
test_mask = df['date'].dt.year >= 2021

print(f"Training set (<= 2020): {train_mask.sum():,} matches ({train_mask.mean():.1%})")
print(f"Test set     (>= 2021): {test_mask.sum():,} matches ({test_mask.mean():.1%})")

# Target distribution in test set
test_dist = df[test_mask]['match_result'].value_counts(normalize=True)
print("\nTest Set Class Distribution:")
for k, v in test_dist.items():
    print(f"  {k:10}: {v:.1%}")


## 3. Execute Model Training & Benchmarking

Train Multinomial Logistic Regression, Random Forest, and Gradient Boosting with balanced sample weights.


In [ ]:
results = train_and_evaluate(split_strategy='temporal')


## 4. Multi-Model Performance Comparison


In [ ]:
eval_summary = evaluate_saved_model(split_strategy='temporal')

comparison_rows = []
for name, m in eval_summary['models'].items():
    comparison_rows.append({
        'Model': name,
        'Accuracy': f"{m['accuracy']:.2%}",
        'Balanced Accuracy': f"{m['balanced_accuracy']:.2%}",
        'Log Loss': f"{m['log_loss']:.4f}",
        'Macro F1': f"{m['macro_f1']:.4f}"
    })

comp_df = pd.DataFrame(comparison_rows)
print("\nBenchmark Comparison Table:")
display(comp_df)


## 5. Feature Importance Analysis

Inspect the strongest predictive drivers from the top-performing model.


In [ ]:
best_pipe = results['gradient_boosting']['model']
gb_model = best_pipe.named_steps['model']
prep = best_pipe.named_steps['preprocessor']

num_names = prep.transformers_[0][2]
cat_encoder = prep.transformers_[1][1]
cat_names = list(cat_encoder.get_feature_names_out(prep.transformers_[1][2]))
all_feature_names = num_names + cat_names

importances = gb_model.feature_importances_
feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=False)

print("Top 15 Feature Importances:")
print(feat_imp.head(15))
